# 01 · Load DANE's monetary-poverty microdata

**Goal of this notebook:** load the `hogares` and `personas` files of *Medición de Pobreza
Monetaria y Desigualdad* for one year, map the variable names, and run sanity checks
before any estimation. Replication of the official rates happens in `02_replicate.ipynb`.

Get the data first: `python data/download.py 2025 --zip <file>` (see `data/download.py`).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from povcol.load import load, HH_KEYS
from povcol.weighted import weighted_rate

YEAR = 2025
APPLY_SPLICE = False   # True only for the *empalme* datasets (ingpcug * fac_emp)

## 1. Load and map variable names
Check the printed mapping against the data dictionary for this year.

In [2]:
hog = load(YEAR, "hogares", apply_splice=APPLY_SPLICE)
per = load(YEAR, "personas")

print("hogares:", hog.shape, "\npersonas:", per.shape)
print("\nColumn mapping used (standard -> original):")
for name, df in [("hogares", hog), ("personas", per)]:
    print(f"  {name}: {df.attrs['column_map']}")

hogares: (276666, 23) 
personas: (773932, 133)

Column mapping used (standard -> original):
  hogares: {'directorio': 'directorio', 'secuencia_p': 'secuencia_p', 'clase': 'clase', 'dpto': 'dpto', 'dominio': 'dominio', 'fex': 'fex_c', 'fex_dpto': 'fex_dpto', 'npobres': 'npobres', 'nindigentes': 'nindigentes', 'ingpcug': 'ingpcug', 'lp': 'lp', 'li': 'li', 'pobre': 'pobre', 'indigente': 'indigente', 'npersug': 'npersug', 'nper': 'nper'}
  personas: {'directorio': 'directorio', 'secuencia_p': 'secuencia_p', 'orden': 'orden', 'clase': 'clase', 'dpto': 'dpto', 'dominio': 'dominio', 'fex': 'fex_c', 'fex_dpto': 'fex_dpto', 'sexo': 'p3271', 'edad': 'p6040', 'parentesco': 'p6050'}


In [3]:
needed = ["fex", "ingpcug", "lp", "li", "pobre", "indigente", "npersug", "clase", "dpto"]
missing = [c for c in needed if c not in hog.columns]
print("Missing in hogares:", missing or "none")
hog[[c for c in needed if c in hog.columns]].describe().T

Missing in hogares: none


,count,mean,std,min,25%,50%,75%,max
fex,276666.0,6.578241e+01,9.161038e+01,1.20405,13.641042,27.640022,8.002339e+01,1.472402e+03
ingpcug,276666.0,1.447343e+06,1.920008e+06,0.00000,491166.666670,908833.333330,1.700000e+06,9.700000e+07
lp,276666.0,5.038195e+05,1.015029e+05,308654.56309,437134.134690,513531.667790,5.872255e+05,6.759192e+05
li,276666.0,2.415234e+05,3.362590e+04,182935.52974,226379.909010,237928.125740,2.595408e+05,3.168152e+05
pobre,276666.0,2.419849e-01,4.282860e-01,0.00000,0.000000,0.000000,0.000000e+00,1.000000e+00
indigente,276666.0,7.809055e-02,2.683145e-01,0.00000,0.000000,0.000000,0.000000e+00,1.000000e+00
npersug,276666.0,2.792237e+00,1.491952e+00,1.00000,2.000000,3.000000,4.000000e+00,1.800000e+01
clase,276666.0,1.134661e+00,3.413613e-01,1.00000,1.000000,1.000000,1.000000e+00,2.000000e+00


## 2. Sanity checks

In [4]:
# Keys: one row per household
dups = hog.duplicated(HH_KEYS).sum()
print(f"Duplicate household keys: {dups}")

# Expansion factors should sum to ~ number of households;
# fex * persons in spending unit should sum to ~ national population (~53 million)
print(f"Sum fex (households):            {hog['fex'].sum():,.0f}")
print(f"Sum fex*npersug (persons in UG): {(hog['fex'] * hog['npersug']).sum():,.0f}")
if "fex" in per.columns:
    print(f"Sum fex over personas file:      {per['fex'].sum():,.0f}")

Duplicate household keys: 0
Sum fex (households):            18,199,756
Sum fex*npersug (persons in UG): 51,511,656
Sum fex over personas file:      51,590,081


In [5]:
# Poverty flags should agree with income vs. lines
chk = pd.DataFrame({
    "pobre_flag": hog["pobre"],
    "pobre_recomputed": (hog["ingpcug"] < hog["lp"]).astype(int),
    "indig_flag": hog["indigente"],
    "indig_recomputed": (hog["ingpcug"] < hog["li"]).astype(int),
})
print("pobre flag == (ingpcug < lp):     ", (chk.pobre_flag == chk.pobre_recomputed).mean())
print("indigente flag == (ingpcug < li): ", (chk.indig_flag == chk.indig_recomputed).mean())
print("\nLines vary by domain; national per-capita lines in the bulletin: lp=482,041, li=236,580")
hog[["lp", "li"]].drop_duplicates().describe()

pobre flag == (ingpcug < lp):      1.0
indigente flag == (ingpcug < li):  1.0

Lines vary by domain; national per-capita lines in the bulletin: lp=482,041, li=236,580


,lp,li
count,50.000000,50.000000
mean,536452.331687,250753.413029
std,79462.483583,30367.585280
min,308654.563090,182935.529740
25%,492455.914950,230770.238608
50%,540393.211860,247371.401100
75%,589884.675950,271561.032997
max,675919.231500,316815.189660


## 3. First look (preliminary, not the replication)

Poverty incidence is a share of **persons**, but `hogares` has one row per household.
So each household is weighted by `fex × npersug` (persons in the spending unit).

In [6]:
w_person = hog["fex"] * hog["npersug"]
print(f"Weighted (persons):   {weighted_rate(hog['pobre'], w_person):.2f}%   official 28.0%")
print(f"Weighted (households):{weighted_rate(hog['pobre'], hog['fex']):.2f}%")
print(f"Unweighted rows:      {weighted_rate(hog['pobre']):.2f}%")

Weighted (persons):   28.04%   official 28.0%
Weighted (households):22.35%
Unweighted rows:      24.20%


Next: `02_replicate.ipynb` — full replication table, extreme poverty, and the weighted vs unweighted comparison.